# Bridge Defect Detection - YOLOv8 Fine-Tuning on CODEBRIM
Author: Lakshan Divakar | Brunel University London

Run each cell top to bottom. ~45 mins on T4 GPU.

**FIRST:** Runtime > Change runtime type > T4 GPU > Save

## Step 1 - Check GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
!nvidia-smi

## Step 2 - Install dependencies

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
print("Ultralytics installed OK")

## Step 3 - Download CODEBRIM dataset
Downloads from Zenodo (~2GB, takes 3-5 mins)

In [ ]:
!wget -q --show-progress -O CODEBRIM.zip \
    "https://zenodo.org/record/2579133/files/CODEBRIM_classification_dataset_v2.zip"
print("Download complete. Extracting...")
!unzip -q CODEBRIM.zip -d CODEBRIM_raw
!ls CODEBRIM_raw/

## Step 4 - Explore dataset structure

In [ ]:
from pathlib import Path
raw_root = Path("CODEBRIM_raw")
for p in sorted(raw_root.rglob("*")):
    if p.is_dir():
        count = len(list(p.glob("*.jpg"))) + len(list(p.glob("*.png")))
        if count > 0:
            print(f"{p}  ({count} images)")

## Step 5 - Convert CODEBRIM to YOLO format

In [ ]:
import shutil, random
from pathlib import Path

CLASSES = ["crack", "spalling", "corrosion", "efflorescence", "exposed_rebar", "background"]
SPLITS  = {"train": 0.70, "val": 0.20, "test": 0.10}
random.seed(42)

raw_root  = Path("CODEBRIM_raw")
yolo_root = Path("dataset")

for split in SPLITS:
    (yolo_root / split / "images").mkdir(parents=True, exist_ok=True)
    (yolo_root / split / "labels").mkdir(parents=True, exist_ok=True)

all_files = []
for cls_id, cls_name in enumerate(CLASSES):
    cls_dirs = [p for p in raw_root.rglob("*") if p.is_dir() and cls_name.lower() in p.name.lower()]
    imgs = []
    for d in cls_dirs:
        imgs += list(d.glob("*.jpg")) + list(d.glob("*.png")) + list(d.glob("*.JPG"))
    if imgs:
        print(f"  [{cls_id}] {cls_name}: {len(imgs)} images")
        for img in imgs:
            all_files.append((img, cls_id))
    else:
        print(f"  [{cls_id}] {cls_name}: NOT FOUND")

random.shuffle(all_files)
n       = len(all_files)
n_train = int(n * 0.70)
n_val   = int(n * 0.20)

split_data = (
    [(f, c, "train") for f, c in all_files[:n_train]] +
    [(f, c, "val")   for f, c in all_files[n_train:n_train+n_val]] +
    [(f, c, "test")  for f, c in all_files[n_train+n_val:]]
)

for img_path, cls_id, split in split_data:
    dst_img = yolo_root / split / "images" / img_path.name
    dst_lbl = yolo_root / split / "labels" / (img_path.stem + ".txt")
    shutil.copy(img_path, dst_img)
    with open(dst_lbl, "w") as f:
        f.write(f"{cls_id} 0.5 0.5 1.0 1.0
")

print(f"Total: {n} | Train: {n_train} | Val: {n_val} | Test: {n-n_train-n_val}")
print("Dataset ready.")

## Step 6 - Write YAML config

In [ ]:
import yaml
from pathlib import Path

yaml_content = {
    "path":  str(Path("dataset").resolve()),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    6,
    "names": ["crack","spalling","corrosion","efflorescence","exposed_rebar","background"]
}
with open("codebrim.yaml", "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)
print("codebrim.yaml written")
!cat codebrim.yaml

## Step 7 - Train YOLOv8 (~30-45 mins on T4 GPU)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data     = "codebrim.yaml",
    epochs   = 50,
    imgsz    = 640,
    batch    = 32,
    device   = 0,
    patience = 15,
    save     = True,
    plots    = True,
    project  = "runs",
    name     = "bridge_defect_v1",
    exist_ok = True,
)
print("Training complete!")

## Step 8 - Evaluate on test set

In [ ]:
from ultralytics import YOLO
best = YOLO("runs/bridge_defect_v1/weights/best.pt")
metrics = best.val(data="codebrim.yaml", split="test")
print(f"mAP@0.5:      {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.3f}")
print(f"Precision:    {metrics.box.mp:.3f}")
print(f"Recall:       {metrics.box.mr:.3f}")

## Step 9 - Visual test on sample image

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import random

best = YOLO("runs/bridge_defect_v1/weights/best.pt")
test_imgs = list(Path("dataset/test/images").glob("*.jpg")) + list(Path("dataset/test/images").glob("*.png"))
sample = random.choice(test_imgs)

results = best.predict(str(sample), conf=0.25, save=False)
annotated = results[0].plot()

fig, axes = plt.subplots(1, 2, figsize=(14,6))
axes[0].imshow(Image.open(sample)); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(annotated); axes[1].set_title("Detected Defects"); axes[1].axis("off")
plt.tight_layout()
plt.savefig("sample_detection.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Sample: {sample.name}")

## Step 10 - Download trained weights to your PC

In [ ]:
import shutil
from google.colab import files
shutil.copy("runs/bridge_defect_v1/weights/best.pt", "bridge_defect_yolov8.pt")
files.download("bridge_defect_yolov8.pt")
files.download("sample_detection.png")
print("Check your browser downloads folder.")